# Evaluation dataset inspection

Compare the Ollila–Malm evaluation candidate with the de Bruyn development dataset using topology and source metadata. This notebook reads the existing files and writes a comparison report.

Sources: [development study](https://doi.org/10.1021/acs.jcim.4c01172), [evaluation study](https://doi.org/10.1038/s42004-026-02155-8), [rat alpha-synuclein reference P37377](https://rest.uniprot.org/uniprotkb/P37377.fasta).

The evaluation role remains provisional. This inspection does not establish independence, adjust sampling weights or compare model predictions.


In [1]:
import json
import re
import struct
import sys
from pathlib import Path

import MDAnalysis as mda
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data/sources.json").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
from inventory import RESIDUE_CODES, verify_archive

manifest_path = ROOT / "data/sources.json"
development_path = ROOT / "data/development_inventory.json"
output_path = ROOT / "data/evaluation_inspection.json"


## Select the source files

The two Zenodo deposits form one evaluation collection. Filename labels are retained as labels; they do not establish independent simulation runs.


In [2]:
manifest = json.loads(manifest_path.read_text())
development = json.loads(development_path.read_text())
dataset = next(item for item in manifest["datasets"] if item["id"] == "ollila_malm")
entries = [(record, item) for record in dataset["records"]
           for item in record["files"] if item["selected"]]
files = {item["name"]: (record, item) for record, item in entries}
if len(files) != len(entries):
    raise ValueError("Duplicate source filenames.")

histidine_aliases = {name: "H" for name in ("HID", "HIE", "HIP", "HSD", "HSE", "HSP")}
codes = RESIDUE_CODES | histidine_aliases
human_sequence = development["trajectories"][0]["observed"]["sequence"]
rat_sequence = (
    "MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVTTVAEKTK"
    "EQVTNVGGAVVTGVTAVAQKTVEGAGNIAAATGFVKKDQMGKGEEGYPQEGILEDMPVDP"
    "SSEAYEMPSEEGYQDYEPEA"
)
print(f"{len(files)} source files; {len(development['trajectories'])} development trajectories.")


50 source files; 2 development trajectories.


## Inspect topology and trajectory pairing

Recheck each TPR checksum and read the first XTC header. The full XTC checksums were verified during downloading and are not rehashed here.

Histidine protonation names are normalised for sequence comparison, while the original residue-50 name is retained. A free N-terminus is inferred only when the terminal atom names contain H1, H2 and H3. Other unrecognised terminal states remain unresolved.

The `not resname SOL` atom count is a candidate subset check. Matching counts do not verify atom order or authorise using that subset as a matching topology.


In [3]:
def inspect_pair(record, xtc, tpr):
    topology_path, trajectory_path = ROOT / tpr["path"], ROOT / xtc["path"]
    checksums = verify_archive(topology_path, tpr)
    if trajectory_path.stat().st_size != xtc["size_bytes"]:
        raise ValueError(f"Trajectory size mismatch: {trajectory_path}")
    with trajectory_path.open("rb") as stream:
        header = stream.read(16)
    if len(header) != 16:
        raise ValueError(f"Truncated XTC header: {trajectory_path}")
    magic, atoms, _, time_ps = struct.unpack(">iiif", header)
    if magic != 1995 or atoms < 1:
        raise ValueError(f"Invalid XTC header: {trajectory_path}")
    universe = mda.Universe(str(topology_path), to_guess=())
    ca = universe.select_atoms("name CA")
    if not len(ca):
        raise ValueError(f"No protein C-alpha atoms: {topology_path}")
    sequence = "".join(codes.get(name, "?") for name in ca.resnames)
    terminal_names = set(ca[0].residue.atoms.names)
    cap_present = "ACE" in universe.residues.resnames
    free_terminus = {"H1", "H2", "H3"} <= terminal_names
    acetylated = True if cap_present else False if free_terminus else None
    residue_129 = next((atom.resname for atom in ca if atom.resid == 129), None)
    phosphorylated = {"SER": False, "SEP": True}.get(residue_129)
    label = re.fullmatch(r"aSyn_replica_(\d+)_(.+)_(\d+)ns\.xtc", xtc["name"])
    if label is None:
        raise ValueError(f"Unrecognised trajectory filename: {xtc['name']}")
    non_water_atoms = int((universe.atoms.resnames != "SOL").sum())
    return {
        "trajectory_id": record["accession"] + "::" + xtc["name"],
        "replica_label_from_filename": label[1],
        "force_field_label_from_filename": label[2],
        "duration_from_filename_ns": int(label[3]),
        "topology_path": tpr["path"], "trajectory_path": xtc["path"],
        "topology_checksums": checksums,
        "trajectory_published_checksum": xtc["checksum"],
        "trajectory_checksum_rechecked": False,
        "protein_residue_count": len(ca), "sequence": sequence,
        "unknown_residue_names": sorted(set(ca.resnames) - codes.keys()),
        "residue_numbering_1_to_140": ca.resids.tolist() == list(range(1, 141)),
        "residue_50_name": next((atom.resname for atom in ca if atom.resid == 50), None),
        "n_terminal_atom_names": sorted(terminal_names),
        "acetyl_cap_present": cap_present, "n_terminal_acetylated": acetylated,
        "residue_129_name": residue_129, "s129_phosphorylated": phosphorylated,
        "matches_rat_reference_P37377": sequence == rat_sequence,
        "development_sequence_differences": [
            f"{a}{i + 1}{b}" for i, (a, b) in enumerate(zip(human_sequence, sequence)) if a != b
        ],
        "topology_atom_count": universe.atoms.n_atoms,
        "xtc_atom_count": atoms, "first_time_ps": time_ps,
        "direct_atom_count_match": universe.atoms.n_atoms == atoms,
        "candidate_subset_rule": "not resname SOL",
        "candidate_subset_atom_count": non_water_atoms,
        "candidate_subset_count_matches": non_water_atoms == atoms,
        "candidate_subset_order_verified": False,
        "development_comparisons": [{
            "trajectory_id": unit["trajectory_id"],
            "sequence_matches": sequence == unit["observed"]["sequence"],
            "acetylation_matches": None if acetylated is None else
                acetylated == unit["observed"]["acetyl_cap_present"],
            "phosphorylation_matches": None if phosphorylated is None else
                phosphorylated == unit["observed"]["s129_phosphorylated"],
            "force_field_label_matches": label[2].replace("-", "").upper() ==
                unit["force_field_folder_label"].replace("-", "").upper(),
        } for unit in development["trajectories"]],
        "unresolved_metadata": {
            "temperature_k": None, "salt_mM": None, "pH": None,
            "original_sample_id": None, "run_seed": None, "batch_id": None,
        },
    }


In [4]:
trajectories = []
for name in sorted(name for name in files if name.endswith(".xtc")):
    if name[:-4] + ".tpr" not in files:
        raise ValueError(f"Missing paired topology for {name}")
    record, xtc = files[name]
    _, tpr = files[name[:-4] + ".tpr"]
    print(f"Inspecting {name}...", flush=True)
    trajectories.append(inspect_pair(record, xtc, tpr))
if not trajectories:
    raise ValueError("No evaluation trajectories found.")
print(f"Inspected {len(trajectories)} evaluation trajectories.")


Inspecting aSyn_replica_01_AMBER03WS_2000ns.xtc...
Inspecting aSyn_replica_01_AMBER99SB-DISP_2000ns.xtc...
Inspecting aSyn_replica_01_AMBER99SBWS_2000ns.xtc...
Inspecting aSyn_replica_01_CHARMM36M_2000ns.xtc...
Inspecting aSyn_replica_01_DESAMBER_2000ns.xtc...
Inspecting aSyn_replica_02_AMBER03WS_2000ns.xtc...
Inspecting aSyn_replica_02_AMBER99SB-DISP_2000ns.xtc...
Inspecting aSyn_replica_02_AMBER99SBWS_2000ns.xtc...
Inspecting aSyn_replica_02_CHARMM36M_2000ns.xtc...
Inspecting aSyn_replica_02_DESAMBER_2000ns.xtc...
Inspecting aSyn_replica_03_AMBER03WS_2000ns.xtc...
Inspecting aSyn_replica_03_AMBER99SB-DISP_2000ns.xtc...
Inspecting aSyn_replica_03_AMBER99SBWS_2000ns.xtc...
Inspecting aSyn_replica_03_CHARMM36M_2000ns.xtc...
Inspecting aSyn_replica_03_DESAMBER_2000ns.xtc...
Inspecting aSyn_replica_04_AMBER03WS_2000ns.xtc...
Inspecting aSyn_replica_04_AMBER99SB-DISP_2000ns.xtc...
Inspecting aSyn_replica_04_AMBER99SBWS_2000ns.xtc...
Inspecting aSyn_replica_04_CHARMM36M_2000ns.xtc...
Inspec

## Compare sequences, modifications and atom counts

A sequence match to a reference identifies the sequence, not the provenance or species of the physical sample. Acetylation and phosphorylation remain separate from normalised sequence identity.


In [5]:
summary = pd.DataFrame([{
    "replica label": row["replica_label_from_filename"],
    "force field label": row["force_field_label_from_filename"],
    "human sequence match": row["sequence"] == human_sequence,
    "rat reference match": row["matches_rat_reference_P37377"],
    "acetylated": row["n_terminal_acetylated"],
    "S129 phosphorylated": row["s129_phosphorylated"],
    "TPR atoms": row["topology_atom_count"],
    "XTC atoms": row["xtc_atom_count"],
    "direct count match": row["direct_atom_count_match"],
    "subset count match": row["candidate_subset_count_matches"],
} for row in trajectories])
display(summary)


,replica label,force field label,human sequence match,rat reference match,acetylated,S129 phosphorylated,TPR atoms,XTC atoms,direct count match,subset count match
0,01,AMBER03WS,True,False,False,False,361023,2355,False,True
1,01,AMBER99SB-DISP,True,False,False,False,358059,2355,False,True
2,01,AMBER99SBWS,True,False,False,False,361023,2355,False,True
3,01,CHARMM36M,True,False,False,False,361027,2355,False,True
4,01,DESAMBER,True,False,False,False,361023,2355,False,True
5,02,AMBER03WS,True,False,False,False,367635,2363,False,True
6,02,AMBER99SB-DISP,True,False,False,False,365755,2363,False,True
7,02,AMBER99SBWS,True,False,False,False,367635,2363,False,True
8,02,CHARMM36M,True,False,False,False,367635,2363,False,True
9,02,DESAMBER,True,False,False,False,367635,2363,False,True


In [6]:
sequence_groups = []
for sequence in sorted({row["sequence"] for row in trajectories}):
    group = [row for row in trajectories if row["sequence"] == sequence]
    sequence_groups.append({
        "replica labels": ", ".join(sorted({row["replica_label_from_filename"] for row in group})),
        "trajectories": len(group),
        "human sequence match": sequence == human_sequence,
        "rat reference match": sequence == rat_sequence,
        "differences from development": ", ".join(group[0]["development_sequence_differences"]) or "None",
    })
display(pd.DataFrame(sequence_groups))


,replica labels,trajectories,human sequence match,rat reference match,differences from development
0,"01, 02",10,True,False,None
1,"03, 04, 05",15,False,True,"A53T, S87N, L100M, N103G, A107Y, D121S, N122S"


## Interpretation and remaining checks

The downloaded collection contains different sequence groups. Labels 01–02 match the development sequence; labels 03–05 have seven differences and match the rat reference sequence P37377.

All inspected evaluation topologies have terminal atom names indicating a free N-terminus. The development structures contain an acetyl cap. The full TPR atom lists also contain water, so they do not directly match the solvent-free XTC files.

The evaluation role remains **unconfirmed**. Temperature, salt, pH, the exact atom ordering after solvent exclusion, and shared trajectories, experimental restraints or upstream fitting still need review. Neither filename labels nor different publications establish statistical independence. These findings have not been used to tune sampling weights.


## Save the inspection report

Keep the original files unchanged. Save the observed metadata, source references, comparisons and unresolved fields to `data/evaluation_inspection.json`.


In [7]:
report = {
    "schema_version": 1,
    "source_study": {
        "dataset_id": dataset["id"], "publication": dataset["publication"],
        "accessions": [record["accession"] for record in dataset["records"]],
    },
    "development_source": development["source_study"],
    "sequence_normalisation": histidine_aliases | {"SEP": "S"},
    "rat_sequence_reference": "https://rest.uniprot.org/uniprotkb/P37377.fasta",
    "trajectory_count": len(trajectories),
    "trajectories": trajectories,
    "evaluation_eligibility": "not_confirmed",
    "limitations": [
        "Only the first XTC header is inspected; complete checksums were verified at download.",
        "A sequence match does not verify sample species or provenance.",
        "Matching subset atom counts do not establish atom-order compatibility.",
        "Temperature, salt and pH have not been recovered from simulation parameters.",
        "Shared trajectories, restraints and upstream fitting require a separate evidence review.",
        "Replica labels and different studies do not establish statistical independence.",
    ],
}
protected = {manifest_path.resolve(), development_path.resolve()}
protected.update((ROOT / item["path"]).resolve() for _, item in entries)
if output_path.resolve() in protected:
    raise ValueError("The report must not overwrite a source file.")
temporary = output_path.with_name(output_path.name + ".tmp")
temporary.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
temporary.replace(output_path)
print(f"Saved {output_path}")


Saved /Users/reecepalmer/Documents/appliedcl/repos/reaSyn/data/evaluation_inspection.json
